# GateGuard — YOLO person/child detector 학습
**실행 전**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

## 1. 환경 설정

In [ ]:
!pip install -q ultralytics pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive 마운트 완료')

In [ ]:
from pathlib import Path
import re

BASE     = Path('/content/drive/MyDrive/ㅈㅎㅊ')
JSON_DIR = BASE / 'json'
ZIP_DIR  = BASE / '지하철 역사 내 CCTV 이상행동 영상' / 'Training' / '개집표기 무단진입'
YOLO_DATASET_DIR = Path('/content/yolo_dataset')
RUNS_DIR         = Path('/content/runs')
DRIVE_SAVE_DIR   = Path('/content/drive/MyDrive/GateGuard_runs')

CLASS_MAP   = {'person': 0, 'child': 1}
CLASS_NAMES = ['person', 'child']

print(f'JSON 폴더 존재  : {JSON_DIR.exists()}')
print(f'ZIP 폴더 존재   : {ZIP_DIR.exists()}')
print()
print('JSON 그룹 목록:')
for p in sorted(JSON_DIR.iterdir()):
    print(f'  {p.name}  ({len(list(p.glob("annotation_*.json")))}개 JSON)')
print()
print('ZIP 파일 목록:')
for p in sorted(ZIP_DIR.iterdir()):
    print(f'  {p.name}')

## 2. YOLO 데이터셋 빌드

In [ ]:
import json, zipfile, yaml, random, re
from pathlib import Path

def to_yolo_line(cid, x, y, w, h, iw, ih):
    cx = max(0.0, min(1.0, (x + w/2) / iw))
    cy = max(0.0, min(1.0, (y + h/2) / ih))
    nw = max(0.0, min(1.0, w / iw))
    nh = max(0.0, min(1.0, h / ih))
    return f'{cid} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}'

def parse_json(path):
    try:
        raw = json.loads(path.read_text(encoding='utf-8'))
    except Exception as e:
        print(f'  [WARN] {path.name}: {e}')
        return None
    meta = raw.get('metadata', {})
    frames = []
    for fr in raw.get('frames', []):
        boxes = []
        for ann in fr.get('annotations', []):
            code = ann['category']['code']
            if code not in CLASS_MAP: continue
            lb = ann['label']
            if lb['width'] > 0 and lb['height'] > 0:
                boxes.append({'cid': CLASS_MAP[code], **lb})
        if boxes:
            frames.append({
                'n': fr['number'],
                'img': fr.get('image', f'frame_{fr["number"]}.jpg'),
                'boxes': boxes
            })
    return {'id': raw['id'], 'w': meta.get('width',3840), 'h': meta.get('height',2160), 'frames': frames}

def get_group_num(name: str):
    m = re.search(r'_(\d+)\.zip$', name)
    return m.group(1) if m else None

def build_dataset(val_ratio=0.2, seed=42):
    random.seed(seed)
    for split in ['train', 'val']:
        (YOLO_DATASET_DIR/'images'/split).mkdir(parents=True, exist_ok=True)
        (YOLO_DATASET_DIR/'labels'/split).mkdir(parents=True, exist_ok=True)

    # ZIP 파일 인덱스: grp → zip_path
    zip_index = {}
    for zp in sorted(ZIP_DIR.glob('*.zip')):
        grp = get_group_num(zp.name)
        if grp:
            zip_index[grp] = zp

    all_videos = []
    for label_dir in sorted(JSON_DIR.iterdir()):
        if not label_dir.is_dir(): continue
        m = re.search(r'_(\d+)$', label_dir.name)
        if not m: continue
        grp = m.group(1)
        if grp not in zip_index:
            print(f'  [SKIP] 그룹 {grp} ZIP 없음')
            continue
        for jp in sorted(label_dir.glob('annotation_*.json')):
            v = parse_json(jp)
            if v and v['frames']:
                all_videos.append((v, zip_index[grp]))

    random.shuffle(all_videos)
    val_n  = max(1, int(len(all_videos) * val_ratio))
    splits = {'val': all_videos[:val_n], 'train': all_videos[val_n:]}
    print(f'총 영상: {len(all_videos)}개  →  train:{len(splits["train"])}  val:{len(splits["val"])}')

    imgs = {'train': 0, 'val': 0}
    cnts = {'train': {0:0, 1:0}, 'val': {0:0, 1:0}}
    skipped = 0

    for split, vid_list in splits.items():
        img_out = YOLO_DATASET_DIR / 'images' / split
        lbl_out = YOLO_DATASET_DIR / 'labels' / split

        # ZIP별로 묶어서 한 번씩만 열기
        from collections import defaultdict
        zip_groups = defaultdict(list)
        for v, zp in vid_list:
            zip_groups[zp].append(v)

        for zp, videos in zip_groups.items():
            print(f'  [{split}] {zp.name} 처리 중...')
            with zipfile.ZipFile(zp) as zf:
                zip_names = set(zf.namelist())
                for v in videos:
                    for fr in v['frames']:
                        inner = f"{v['id']}/{fr['img']}"
                        if inner not in zip_names:
                            skipped += 1
                            continue
                        stem = f"{v['id']}_{fr['n']:06d}"
                        (img_out / f'{stem}.jpg').write_bytes(zf.read(inner))
                        lines = [to_yolo_line(b['cid'],b['x'],b['y'],b['width'],b['height'],v['w'],v['h'])
                                 for b in fr['boxes']]
                        (lbl_out / f'{stem}.txt').write_text('\n'.join(lines))
                        for b in fr['boxes']: cnts[split][b['cid']] += 1
                        imgs[split] += 1

    yaml_path = YOLO_DATASET_DIR / 'data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump({'path': str(YOLO_DATASET_DIR), 'train': 'images/train', 'val': 'images/val',
                   'nc': len(CLASS_NAMES), 'names': CLASS_NAMES}, f, allow_unicode=True, sort_keys=False)

    for s in ['train', 'val']:
        print(f'[{s}] 이미지:{imgs[s]}장  person:{cnts[s][0]}  child:{cnts[s][1]}')
    if skipped: print(f'건너뜀: {skipped}')
    print(f'\n{yaml_path.read_text()}')

build_dataset(val_ratio=0.2)

## 3. 학습

In [ ]:
# 처음엔 QUICK_MODE=True → 이상없으면 False로 본 학습
QUICK_MODE = True
MODEL      = 'yolo11n.pt'

EPOCHS   = 5   if QUICK_MODE else 30
IMGSZ    = 416 if QUICK_MODE else 640
BATCH    = 8   if QUICK_MODE else -1
CACHE    = False if QUICK_MODE else True
RUN_NAME = 'gateguard_quick' if QUICK_MODE else 'gateguard_detector'

train_n = len(list((YOLO_DATASET_DIR/'images'/'train').glob('*.jpg')))
val_n   = len(list((YOLO_DATASET_DIR/'images'/'val').glob('*.jpg')))
assert train_n > 0, '❌ train 이미지 없음 — 셀 2 먼저 실행하세요'
print(f'train:{train_n}장  val:{val_n}장')
print(f'모델:{MODEL}  epochs:{EPOCHS}  imgsz:{IMGSZ}  batch:{BATCH}')

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=str(YOLO_DATASET_DIR / 'data.yaml'),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, cache=CACHE,
    project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True,
)
best_pt = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
print(f'학습 완료: {best_pt}')

## 4. 결과 확인

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg

run_dir = RUNS_DIR / RUN_NAME
for name in ['results.png', 'confusion_matrix.png']:
    p = run_dir / name
    if p.exists():
        fig, ax = plt.subplots(figsize=(12,5))
        ax.imshow(mpimg.imread(p)); ax.axis('off'); ax.set_title(name)
        plt.tight_layout(); plt.show()

In [ ]:
best_model = YOLO(str(best_pt))
metrics = best_model.val(data=str(YOLO_DATASET_DIR / 'data.yaml'))
print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP50-95 : {metrics.box.map:.4f}')
for i, name in enumerate(CLASS_NAMES):
    if i < len(metrics.box.ap50):
        print(f'  {name} AP50: {metrics.box.ap50[i]:.4f}')

In [ ]:
import random
from PIL import Image

val_imgs = list((YOLO_DATASET_DIR/'images'/'val').glob('*.jpg'))
if val_imgs:
    sample = random.choice(val_imgs)
    r = best_model.predict(str(sample), conf=0.3, verbose=False)[0]
    fig, axes = plt.subplots(1,2,figsize=(16,6))
    axes[0].imshow(Image.open(sample)); axes[0].set_title('원본'); axes[0].axis('off')
    axes[1].imshow(r.plot()[:,:,::-1]); axes[1].set_title(f'탐지 {len(r.boxes)}건'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

## 5. Drive에 결과 저장

In [ ]:
import shutil
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
dest = DRIVE_SAVE_DIR / RUN_NAME
if dest.exists(): shutil.rmtree(dest)
shutil.copytree(run_dir, dest)
print(f'저장 완료: {dest}')
print(f'best.pt  : {dest / "weights" / "best.pt"}')